# Eval: Region Plotting (Working Demo)

This notebook supports both modes:
- use a real `predictions.nc` if available
- otherwise auto-fallback to a synthetic demo dataset


In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from eval.region_plotting.local_plotting import get_region_ds, plot_x_y

RNG = np.random.default_rng(42)
TMP = Path('/tmp/region_plotting_demo')
TMP.mkdir(parents=True, exist_ok=True)
print('Output dir:', TMP)


In [ ]:
# Step 1: choose real predictions path (edit this), fallback to synthetic if missing/unreadable
real_predictions_nc = Path('/home/ecm5702/perm/eval/manual_o320r2/eval/predictions_20230829_step048/predictions.nc')

def build_synthetic_dataset():
    n_points = 2400
    weather_states = np.array(['10u', '10v', '2t', 'msl'])

    lon_hres = RNG.uniform(-130, 40, size=n_points).astype(np.float32)
    lat_hres = RNG.uniform(-10, 60, size=n_points).astype(np.float32)
    lon_lres = RNG.uniform(-130, 40, size=n_points // 3).astype(np.float32)
    lat_lres = RNG.uniform(-10, 60, size=n_points // 3).astype(np.float32)

    shape_h = (1, 1, n_points, len(weather_states))
    shape_l = (1, 1, n_points // 3, len(weather_states))

    base_h = RNG.normal(size=shape_h).astype(np.float32)
    base_l = RNG.normal(size=shape_l).astype(np.float32)

    ds_local = xr.Dataset(
        data_vars={
            'x_0': (['sample','ensemble_member','grid_point_hres','weather_state'], base_h),
            'y_0': (['sample','ensemble_member','grid_point_hres','weather_state'], base_h * 0.8),
            'y_1': (['sample','ensemble_member','grid_point_hres','weather_state'], base_h * 1.2),
            'y_pred_0': (['sample','ensemble_member','grid_point_hres','weather_state'], base_h * 1.05),
            'y_pred_1': (['sample','ensemble_member','grid_point_hres','weather_state'], base_h * 0.95),
            'x': (['sample','ensemble_member','grid_point_lres','weather_state'], base_l),
        },
        coords={
            'sample': np.array([0]),
            'ensemble_member': np.array([0]),
            'grid_point_hres': np.arange(n_points),
            'grid_point_lres': np.arange(n_points // 3),
            'weather_state': weather_states,
            'lon_hres': ('grid_point_hres', lon_hres),
            'lat_hres': ('grid_point_hres', lat_hres),
            'lon_lres': ('grid_point_lres', lon_lres),
            'lat_lres': ('grid_point_lres', lat_lres),
        },
        attrs={'grid': 'O320'},
    )

    for var in ['x_0','y_0','y_1','y_pred_0','y_pred_1']:
        ds_local[var].attrs['lon'] = 'lon_hres'
        ds_local[var].attrs['lat'] = 'lat_hres'
    return ds_local

if real_predictions_nc.exists():
    try:
        ds = xr.open_dataset(real_predictions_nc)
        print('Using real predictions file:', real_predictions_nc)
    except Exception as e:
        print('Real predictions exists but is unreadable; using synthetic fallback.')
        print('Error:', type(e).__name__, e)
        ds = build_synthetic_dataset()
else:
    print('Real predictions file not found; using synthetic fallback:', real_predictions_nc)
    ds = build_synthetic_dataset()

print(ds)


In [ ]:
# Step 2: select one predefined region
region_name = 'rocky_mountains'
region_ds = get_region_ds(ds, region_name)
print('Region:', region_name)
print('Region box:', region_ds.attrs.get('region'))
print('Region sizes:', dict(region_ds.sizes))


In [ ]:
# Step 3: draw inline region comparison plot
model_vars = ['x_0', 'y_0', 'y_pred_0', 'x', 'y', 'y_pred']
model_vars = [v for v in model_vars if v in region_ds.data_vars]
states = ['10u', '2t', 'msl']
states = [s for s in states if s in region_ds.weather_state.values]

sample_val = region_ds.sample.values[0] if 'sample' in region_ds.coords else 0
member_val = region_ds.ensemble_member.values[0] if 'ensemble_member' in region_ds.coords else 0
ds_sample = region_ds.sel(sample=sample_val, ensemble_member=member_val)

fig = plot_x_y(
    ds_sample=ds_sample,
    list_model_variables=model_vars,
    weather_states=states,
    title=f'{region_name} sample',
)
plt.show()


In [ ]:
# Step 4: save plot artifact
out_png = TMP / 'region_plotting_demo.png'
fig.savefig(out_png, dpi=150, bbox_inches='tight')
print('Saved:', out_png)


In [ ]:
# Step 5: save dataset copy (useful for quick local tests)
out_nc = TMP / 'predictions_selected_or_synthetic.nc'
if out_nc.exists():
    out_nc.unlink()
ds.to_netcdf(out_nc)
print('Saved dataset copy:', out_nc)
